# FTIR ML Solver v3 — Colab Training

## Setup
Go to **Runtime → Change runtime type → GPU (T4)** before running.

> ⏱ Estimate: ~2 min/epoch on T4 with 10 000 synthetic + 557 reference samples.  
> For 100 epochs with 50 000 synthetic samples use the Full Training cell below.

In [ ]:
# Cell 1 — Clone repo and install
# GIT_TERMINAL_PROMPT=0 prevents git from hanging waiting for a password prompt
import os
os.environ['GIT_TERMINAL_PROMPT'] = '0'

!git clone https://github.com/DrSuppe/FTIR_absorbtion_ML.git /content/ftir
os.chdir('/content/ftir')
print('Working directory:', os.getcwd())
!pip install -e . -q

In [ ]:
# Cell 2 — Upload reference spectra (not in repo — binary files)
# On your Mac, first zip them:
#   cd .../FTIR_ML_fitting
#   zip -r spc_files.zip reference_spectra/spc_files/
# Then run this cell and choose that zip file.
import os
from google.colab import files

uploaded = files.upload()  # pick spc_files.zip
for fname in uploaded:
    if fname.endswith('.zip'):
        os.makedirs('reference_spectra', exist_ok=True)
        os.system(f'unzip -q "{fname}" -d reference_spectra/')
        print(f'Unzipped {fname}')

# Rebuild manifest from the uploaded SPC files
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f'Manifest ready: {len(m)} spectra, {m.species.nunique()} species')

In [ ]:
# Cell 3 — Quick smoke test (~10 min on T4)
!python3 train.py \
    --n-synthetic 5000 \
    --epochs 10 \
    --batch-size 64 \
    --log-every 2

In [ ]:
# Cell 4 — Full training run (T4, ~1-2 hrs)
# Uncomment when ready.
# !python3 train.py \
#     --n-synthetic 50000 \
#     --epochs 100 \
#     --batch-size 128 \
#     --lr 3e-4 \
#     --warmup-epochs 5 \
#     --log-every 5

In [ ]:
# Cell 5 — [Optional] Save checkpoint to Google Drive (survives session timeouts)
# from google.colab import drive
# drive.mount('/content/drive')
# Then add:  --checkpoint-dir /content/drive/MyDrive/ftir_checkpoints  to your train.py call

In [ ]:
# Cell 6 — View the latest per-species MAE plot
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.image as mpimg

plots = sorted(Path('reports').glob('mae_per_species_*.png'))
if plots:
    img = mpimg.imread(plots[-1])
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'MAE — {plots[-1].name}')
    plt.show()
else:
    print('No MAE plots yet — run training first.')

In [ ]:
# Cell 7 — Download best checkpoint to your local machine
from google.colab import files
files.download('checkpoints/best_model.pt')